# HSI Checkpoint Diagnostics

- 加载 HSI inpainting checkpoint、config 和重建结果
- 在第 3 个单元选择显示模式：single_band 或 pseudorgb
- single_band 模式下对比 GT / Mask / Observed / Reconstruction / Error
- pseudorgb 模式下按三条 band 构造 pseudoRGB，对比 GT / Observed / Reconstruction
- 计算整图指标，并结合当前显示模式输出相关信息
- 分析 abundance、E0、E_hat、delta_E、U/V/gamma 等参数分布
- 可视化高权重点空间位置，以及变化最大的 endmember 光谱

使用方式：
1. 先在第 3 个单元设置 checkpoint_path 和 display_mode
2. 如果选择 single_band，就填写 channel_index
3. 如果选择 pseudorgb，就填写 pseudorgb_bands
4. 顺序运行前 5 个单元完成模型与数据加载
5. 再运行后续单元做显示和参数数值分析

In [ ]:
from pathlib import Path
import sys
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "inpainting_train_hsi.py").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "gsplat") not in sys.path:
    sys.path.insert(0, str(repo_root / "gsplat"))

from gaussianimage_cholesky_hsi import GaussianImage_Cholesky_HSI
from inpainting_train_hsi import load_hsi_dataset
from inpainting_utils import (
    compute_coverage_map,
    compute_error_map,
    compute_inpainting_psnrs,
    compute_region_error_map,
    compute_ssim_hsi,
    generate_mask,
    get_missing_mask,
)

plt.style.use("default")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"repo_root: {repo_root}")
print(f"device: {device}")
if device.type != "cuda":
    print("Warning: this notebook expects a CUDA-capable gsplat build for forward rendering.")

In [ ]:
checkpoint_path = repo_root / "checkpoints_inpainting_hsi" / "your_run" / "gaussian_model.best.pth.tar"
config_path = checkpoint_path.parent / "config.yaml"

# 可选：手动覆盖数据集名；默认从 config.yaml 读取
dataset_override = None

# 显示模式："single_band" 或 "pseudorgb"
display_mode = "single_band"

# single_band 模式使用的通道编号
channel_index = 0

# pseudorgb 模式使用的 band 顺序：[blue, green, red]
# 设为 None 时，会根据通道数自动选择三条 band
pseudorgb_bands = None

# 可选：指定一个像素看整条光谱；如果为 None，则自动选择参考误差最大的像素
pixel_y = None
pixel_x = None

# Centers Colored by Mean Weight 只显示高权重点
# 仅显示 point_strength 位于前 (1 - center_weight_quantile) 的点
center_weight_quantile = 0.9

# 分析参数
topk_points = 20
topk_endmembers = 4
hist_bins = 60
scatter_alpha = 0.45
scatter_size = 10

assert checkpoint_path.exists(), f"Checkpoint not found: {checkpoint_path}"
assert config_path.exists(), f"Config not found: {config_path}"

In [ ]:
def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


def load_checkpoint_state(path, device):
    checkpoint = torch.load(path, map_location=device)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint and "E0" not in checkpoint:
        checkpoint = checkpoint["state_dict"]
    return checkpoint


def set_all_seeds(seed):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


def resolve_dataset_name(config):
    dataset_name = dataset_override or config.get("dataset")
    if dataset_name is None:
        raise ValueError("dataset is missing in config.yaml and dataset_override is None")
    return dataset_name


def safe_display_mode(mode):
    mode = str(mode).strip().lower()
    if mode not in {"single_band", "pseudorgb"}:
        raise ValueError(f"display_mode must be 'single_band' or 'pseudorgb', got {mode}")
    return mode


def safe_channel_index(index, num_channels):
    index = int(index)
    if index < 0 or index >= num_channels:
        raise ValueError(f"channel_index must be in [0, {num_channels - 1}], got {index}")
    return index


def auto_pseudorgb_bands(num_channels):
    blue = max(0, int(round(0.2 * (num_channels - 1))))
    green = max(0, int(round(0.5 * (num_channels - 1))))
    red = max(0, int(round(0.8 * (num_channels - 1))))
    return [blue, green, red]


def safe_pseudorgb_bands(bands, num_channels):
    if bands is None:
        return auto_pseudorgb_bands(num_channels)
    if len(bands) != 3:
        raise ValueError(f"pseudorgb_bands must contain exactly 3 integers, got {bands}")
    validated = [int(b) for b in bands]
    for b in validated:
        if b < 0 or b >= num_channels:
            raise ValueError(f"PseudoRGB band {b} is out of range for {num_channels} channels")
    return validated


def tensor_to_2d_numpy(tensor):
    if tensor.dim() == 4:
        tensor = tensor.squeeze(0)
    if tensor.dim() == 3 and tensor.shape[0] == 1:
        tensor = tensor.squeeze(0)
    return tensor.detach().float().cpu().numpy()


def tensor_to_hwc_numpy(tensor):
    if tensor.dim() == 4:
        tensor = tensor.squeeze(0)
    tensor = tensor.detach().float().cpu()
    if tensor.dim() == 2:
        return tensor.numpy()
    if tensor.dim() == 3:
        return tensor.permute(1, 2, 0).numpy()
    raise ValueError(f"Unsupported tensor shape for HWC conversion: {tuple(tensor.shape)}")


def create_pseudorgb(image, bands):
    blue = image[:, :, bands[0]]
    green = image[:, :, bands[1]]
    red = image[:, :, bands[2]]

    def normalize(channel):
        channel_min = np.min(channel)
        channel_max = np.max(channel)
        return (channel - channel_min) / (channel_max - channel_min + 1e-8)

    blue_norm = normalize(blue)
    green_norm = normalize(green)
    red_norm = normalize(red)
    return np.dstack((red_norm, green_norm, blue_norm))


def channel_slice(tensor, channel):
    return tensor[:, channel:channel + 1]


def channel_mask(mask, channel):
    if mask.shape[1] == 1:
        return mask
    return mask[:, channel:channel + 1]


def compute_channel_metrics(pred, target, observed_mask, channel):
    pred_c = channel_slice(pred, channel)
    target_c = channel_slice(target, channel)
    observed_mask_c = channel_mask(observed_mask, channel)
    return compute_inpainting_psnrs(pred_c, target_c, observed_mask_c)


def build_gt_image(config, device):
    dataset_name = resolve_dataset_name(config)
    I_np = load_hsi_dataset(dataset_name)
    gt_image = torch.tensor(I_np, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
    gt_image = torch.clamp(gt_image, 0, 1).to(device)
    return gt_image, dataset_name


def build_observed_mask(config, gt_image, device):
    set_all_seeds(int(config.get("seed", 42)))
    mask_type = config.get("mask_type", "random")
    num_channels = gt_image.shape[1] if mask_type == "elementwise" else 1
    return generate_mask(
        gt_image.shape[-2],
        gt_image.shape[-1],
        mask_type=mask_type,
        mask_ratio=float(config.get("mask_ratio", 0.5)),
        block_size=int(config.get("block_size", 64)),
        num_blocks=int(config.get("num_blocks", 4)),
        C=num_channels,
        device=device,
    )


def build_model_from_checkpoint(config, gt_image, checkpoint_state, checkpoint_path, device):
    e0_tensor = checkpoint_state.get("E0")
    if e0_tensor is None:
        e0_path = checkpoint_path.parent / "E0_maskedNMF.npy"
        if not e0_path.exists():
            raise KeyError("E0 not found in checkpoint state_dict and E0_maskedNMF.npy is missing")
        e0_init = np.load(e0_path)
    else:
        e0_init = e0_tensor.detach().cpu().numpy()

    rank = int(config.get("rank", e0_init.shape[0]))
    model = GaussianImage_Cholesky_HSI(
        loss_type=config.get("loss_type", "L2"),
        opt_type=config.get("opt_type", "adan"),
        num_points=int(config["num_points"]),
        H=gt_image.shape[-2],
        W=gt_image.shape[-1],
        rank=rank,
        C=gt_image.shape[1],
        E=e0_init,
        calib_rank=int(config.get("calib_rank", 2)),
        gamma=float(config.get("gamma", 0.1)),
        freeze_endmember_calibration=bool(config.get("freeze_endmember_calibration", False)),
        BLOCK_H=16,
        BLOCK_W=16,
        device=device,
        lr=float(config.get("lr", 5e-3)),
    ).to(device)

    model_dict = model.state_dict()
    filtered = {k: v for k, v in checkpoint_state.items() if k in model_dict}
    model_dict.update(filtered)
    model.load_state_dict(model_dict)
    model.eval()
    return model


def summarize_tensor(name, tensor):
    flat = tensor.detach().float().reshape(-1).cpu()
    return {
        "name": name,
        "shape": tuple(tensor.shape),
        "mean": float(flat.mean().item()),
        "std": float(flat.std(unbiased=False).item()),
        "min": float(flat.min().item()),
        "max": float(flat.max().item()),
    }


def effective_gamma_value(model):
    if getattr(model, "freeze_endmember_calibration", False):
        return 0.0
    return float((model.max_calib_scale * torch.tanh(model.calib_gamma.detach())).item())


def effective_parameter_views(model, abundance, E_hat):
    delta_E = E_hat.detach().cpu() - model.E0.detach().cpu()
    point_strength = model.get_features.detach().cpu().mean(dim=1)
    return {
        "xyz_tanh": model.get_xyz.detach().cpu(),
        "cholesky_effective": model.get_cholesky_elements.detach().cpu(),
        "features_dc": model.get_features.detach().cpu(),
        "E0": model.E0.detach().cpu(),
        "E_hat": E_hat.detach().cpu(),
        "delta_E": delta_E,
        "calib_U": model.calib_U.detach().cpu(),
        "calib_V": model.calib_V.detach().cpu(),
        "abundance": abundance.detach().cpu(),
        "point_strength": point_strength,
    }

In [ ]:
config = load_yaml(config_path)
checkpoint_state = load_checkpoint_state(checkpoint_path, device)

gt_image, dataset_name = build_gt_image(config, device)
display_mode = safe_display_mode(display_mode)
channel_index = safe_channel_index(channel_index, gt_image.shape[1])
pseudorgb_bands = safe_pseudorgb_bands(pseudorgb_bands, gt_image.shape[1])

observed_mask = build_observed_mask(config, gt_image, device)
missing_mask = get_missing_mask(observed_mask)
coverage_map = compute_coverage_map(observed_mask)
observed_image = gt_image * observed_mask

model = build_model_from_checkpoint(config, gt_image, checkpoint_state, checkpoint_path, device)
with torch.no_grad():
    outputs = model.forward()
    reconstruction = outputs["render"]
    abundance = outputs["abundance"]
    E_hat = outputs["E_hat"]

E0 = model.E0.detach()
delta_E = E_hat - E0
overall_metrics = compute_inpainting_psnrs(reconstruction, gt_image, observed_mask)
ssim_value = compute_ssim_hsi(reconstruction, gt_image)
delta_norm = model.get_delta_E_norm()
gamma_value = effective_gamma_value(model)

gt_hwc = tensor_to_hwc_numpy(gt_image)
observed_hwc = tensor_to_hwc_numpy(observed_image)
reconstruction_hwc = tensor_to_hwc_numpy(reconstruction)

channel_metrics = None
selected_gt = None
selected_mask = None
selected_observed = None
selected_reconstruction = None
selected_full_error = None
selected_missing_error = None
pseudorgb_gt = None
pseudorgb_observed = None
pseudorgb_reconstruction = None
pseudorgb_abs_error = None

if display_mode == "single_band":
    selected_gt = channel_slice(gt_image, channel_index)
    selected_mask = channel_mask(observed_mask, channel_index)
    selected_observed = channel_slice(observed_image, channel_index)
    selected_reconstruction = channel_slice(reconstruction, channel_index)
    selected_full_error = compute_error_map(selected_reconstruction, selected_gt)
    selected_missing_error = compute_region_error_map(
        selected_reconstruction,
        selected_gt,
        get_missing_mask(selected_mask),
    )
    channel_metrics = compute_channel_metrics(reconstruction, gt_image, observed_mask, channel_index)
    reference_error = selected_full_error
else:
    pseudorgb_gt = create_pseudorgb(gt_hwc, pseudorgb_bands)
    pseudorgb_observed = create_pseudorgb(observed_hwc, pseudorgb_bands)
    pseudorgb_reconstruction = create_pseudorgb(reconstruction_hwc, pseudorgb_bands)
    pseudorgb_abs_error = np.abs(pseudorgb_reconstruction - pseudorgb_gt)
    reference_error = (reconstruction - gt_image).abs().mean(dim=1, keepdim=True)

if pixel_y is None or pixel_x is None:
    flat_index = int(reference_error.reshape(-1).argmax().item())
    pixel_y = flat_index // gt_image.shape[-1]
    pixel_x = flat_index % gt_image.shape[-1]
else:
    pixel_y = int(pixel_y)
    pixel_x = int(pixel_x)

saved_artifacts = {
    "config": config_path.exists(),
    "E0_maskedNMF.npy": (checkpoint_path.parent / "E0_maskedNMF.npy").exists(),
    "E_hat_calibrated.npy": (checkpoint_path.parent / "E_hat_calibrated.npy").exists(),
    "abundance.npy": (checkpoint_path.parent / "abundance.npy").exists(),
}

print(f"checkpoint_path: {checkpoint_path}")
print(f"dataset: {dataset_name}")
print(f"tensor shape: {tuple(gt_image.shape)}")
print(f"mask_type: {config.get('mask_type')} | mask_ratio: {config.get('mask_ratio')}")
print(f"display_mode: {display_mode}")
if display_mode == "single_band":
    print(f"selected channel: {channel_index}")
    print(f"Selected-channel PSNR(full): {channel_metrics['psnr_full']:.4f}")
    print(f"Selected-channel PSNR(observed-only): {channel_metrics['psnr_observed']:.4f}")
    print(f"Selected-channel PSNR(missing-only): {channel_metrics['psnr_missing']:.4f}")
else:
    print(f"selected pseudoRGB bands [B, G, R]: {pseudorgb_bands}")
print(f"overall observed ratio: {observed_mask.mean().item():.6f}")
print(f"PSNR(full): {overall_metrics['psnr_full']:.4f}")
print(f"PSNR(observed-only): {overall_metrics['psnr_observed']:.4f}")
print(f"PSNR(missing-only): {overall_metrics['psnr_missing']:.4f}")
print(f"SSIM(mean over channels): {ssim_value:.6f}")
print(f"effective gamma: {gamma_value:.6f}")
print(f"||delta_E||_F: {delta_norm:.6f}")
print(f"spectrum pixel (y, x): ({pixel_y}, {pixel_x})")
print(f"saved artifacts: {saved_artifacts}")

In [ ]:
if display_mode != "single_band":
    print("display_mode is pseudorgb, so the single-band visualization cell is skipped.")
else:
    vis_items = [
        (f"GT | channel {channel_index}", tensor_to_2d_numpy(selected_gt), "viridis", 0.0, 1.0),
        (f"Mask | channel {channel_index}", tensor_to_2d_numpy(selected_mask), "gray", 0.0, 1.0),
        (f"Observed | channel {channel_index}", tensor_to_2d_numpy(selected_observed), "viridis", 0.0, 1.0),
        (f"Reconstruction | channel {channel_index}", tensor_to_2d_numpy(selected_reconstruction), "viridis", 0.0, 1.0),
        (f"Full Abs Error | channel {channel_index}", tensor_to_2d_numpy(selected_full_error), "inferno", 0.0, float(selected_full_error.max().item()) + 1e-8),
        (f"Missing-only Error | channel {channel_index}", tensor_to_2d_numpy(selected_missing_error), "inferno", 0.0, float(selected_missing_error.max().item()) + 1e-8),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, (title, img, cmap, vmin, vmax) in zip(axes.flat, vis_items):
        ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    if observed_mask.shape[1] > 1:
        plt.figure(figsize=(6, 5))
        plt.imshow(tensor_to_2d_numpy(coverage_map), cmap="magma", vmin=0.0, vmax=1.0)
        plt.title("Coverage Map Across Channels")
        plt.colorbar()
        plt.axis("off")
        plt.show()

    channel_summary_df = pd.DataFrame([
        {"item": "gt", "mean": float(selected_gt.mean().item()), "std": float(selected_gt.std(unbiased=False).item()), "min": float(selected_gt.min().item()), "max": float(selected_gt.max().item())},
        {"item": "mask", "mean": float(selected_mask.mean().item()), "std": float(selected_mask.std(unbiased=False).item()), "min": float(selected_mask.min().item()), "max": float(selected_mask.max().item())},
        {"item": "observed", "mean": float(selected_observed.mean().item()), "std": float(selected_observed.std(unbiased=False).item()), "min": float(selected_observed.min().item()), "max": float(selected_observed.max().item())},
        {"item": "reconstruction", "mean": float(selected_reconstruction.mean().item()), "std": float(selected_reconstruction.std(unbiased=False).item()), "min": float(selected_reconstruction.min().item()), "max": float(selected_reconstruction.max().item())},
        {"item": "abs_error", "mean": float(selected_full_error.mean().item()), "std": float(selected_full_error.std(unbiased=False).item()), "min": float(selected_full_error.min().item()), "max": float(selected_full_error.max().item())},
    ])
    display(channel_summary_df)

In [ ]:
if display_mode != "pseudorgb":
    print("display_mode is single_band, so the pseudoRGB visualization cell is skipped.")
else:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    pseudorgb_items = [
        ("PseudoRGB GT", pseudorgb_gt),
        ("PseudoRGB Observed", pseudorgb_observed),
        ("PseudoRGB Reconstruction", pseudorgb_reconstruction),
        ("PseudoRGB Abs Error", np.clip(pseudorgb_abs_error / (pseudorgb_abs_error.max() + 1e-8), 0.0, 1.0)),
    ]

    for ax, (title, img) in zip(axes, pseudorgb_items):
        ax.imshow(np.clip(img, 0.0, 1.0))
        ax.set_title(f"{title}\nBands [B,G,R] = {pseudorgb_bands}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    if observed_mask.shape[1] > 1:
        plt.figure(figsize=(6, 5))
        plt.imshow(tensor_to_2d_numpy(coverage_map), cmap="magma", vmin=0.0, vmax=1.0)
        plt.title("Coverage Map Across Channels")
        plt.colorbar()
        plt.axis("off")
        plt.show()

In [ ]:
param_views = effective_parameter_views(model, abundance, E_hat)
stats_rows = [
    summarize_tensor(name, tensor)
    for name, tensor in param_views.items()
    if name != "point_strength"
]
stats_df = pd.DataFrame(stats_rows).sort_values("name").reset_index(drop=True)
display(stats_df)

point_strength = param_views["point_strength"]
topk = min(topk_points, point_strength.numel())
top_idx = torch.topk(point_strength, topk).indices
xy_top = model.get_xyz.detach().cpu()[top_idx]

top_points_df = pd.DataFrame({
    "point_index": top_idx.numpy(),
    "mean_feature_strength": point_strength[top_idx].numpy(),
    "x_norm": xy_top[:, 0].numpy(),
    "y_norm": xy_top[:, 1].numpy(),
})
display(top_points_df)

delta_row_norm = delta_E.detach().cpu().norm(dim=1)
endmember_df = pd.DataFrame({
    "endmember_index": np.arange(delta_E.shape[0]),
    "E0_mean": E0.detach().cpu().mean(dim=1).numpy(),
    "E_hat_mean": E_hat.detach().cpu().mean(dim=1).numpy(),
    "delta_norm": delta_row_norm.numpy(),
    "delta_max_abs": delta_E.detach().cpu().abs().max(dim=1).values.numpy(),
}).sort_values("delta_norm", ascending=False).reset_index(drop=True)
display(endmember_df.head(topk_endmembers))

per_channel_mse = ((reconstruction - gt_image) ** 2).mean(dim=(0, 2, 3)).detach().cpu().numpy()
observed_ratio_per_channel = (
    observed_mask.mean(dim=(0, 2, 3)).detach().cpu().numpy()
    if observed_mask.shape[1] > 1
    else np.full(gt_image.shape[1], float(observed_mask.mean().item()))
)
delta_col_norm = delta_E.detach().cpu().norm(dim=0).numpy()
channel_df = pd.DataFrame({
    "channel": np.arange(gt_image.shape[1]),
    "mse": per_channel_mse,
    "psnr": 10 * np.log10(1.0 / np.maximum(per_channel_mse, 1e-12)),
    "observed_ratio": observed_ratio_per_channel,
    "delta_E_col_norm": delta_col_norm,
}).sort_values("mse", ascending=False).reset_index(drop=True)
display(channel_df.head(15))

scalar_df = pd.DataFrame([
    {"metric": "effective_gamma", "value": gamma_value},
    {"metric": "delta_E_fro_norm", "value": delta_norm},
    {"metric": "observed_ratio_mean", "value": float(observed_mask.mean().item())},
    {"metric": "selected_channel", "value": float(channel_index)},
    {"metric": "spectrum_pixel_y", "value": float(pixel_y)},
    {"metric": "spectrum_pixel_x", "value": float(pixel_x)},
])
display(scalar_df)

In [ ]:
plot_groups = {
    "xyz_tanh": param_views["xyz_tanh"].reshape(-1).numpy(),
    "cholesky_effective": param_views["cholesky_effective"].reshape(-1).numpy(),
    "features_dc": param_views["features_dc"].reshape(-1).numpy(),
    "calib_U": param_views["calib_U"].reshape(-1).numpy(),
    "calib_V": param_views["calib_V"].reshape(-1).numpy(),
    "E0": param_views["E0"].reshape(-1).numpy(),
    "E_hat": param_views["E_hat"].reshape(-1).numpy(),
    "delta_E": param_views["delta_E"].reshape(-1).numpy(),
    "abundance": param_views["abundance"].reshape(-1).numpy(),
    "point_strength": param_views["point_strength"].reshape(-1).numpy(),
}

fig, axes = plt.subplots(len(plot_groups), 2, figsize=(14, 3 * len(plot_groups)))
for row_idx, (name, values) in enumerate(plot_groups.items()):
    axes[row_idx, 0].hist(values, bins=hist_bins, color="tab:blue", alpha=0.85)
    axes[row_idx, 0].set_title(f"{name} histogram")
    axes[row_idx, 0].grid(alpha=0.2)

    axes[row_idx, 1].boxplot(values, vert=False)
    axes[row_idx, 1].set_title(f"{name} boxplot")
    axes[row_idx, 1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
xy = model.get_xyz.detach().cpu()
x_px = ((xy[:, 0] + 1.0) * 0.5 * (gt_image.shape[-1] - 1)).numpy()
y_px = ((xy[:, 1] + 1.0) * 0.5 * (gt_image.shape[-2] - 1)).numpy()
strength = point_strength.numpy()

weight_threshold = float(np.quantile(strength, center_weight_quantile))
high_weight_mask = strength >= weight_threshold
if not np.any(high_weight_mask):
    high_weight_mask = np.ones_like(strength, dtype=bool)

x_px_high = x_px[high_weight_mask]
y_px_high = y_px[high_weight_mask]
strength_high = strength[high_weight_mask]

if display_mode == "single_band":
    background_gt = tensor_to_2d_numpy(channel_slice(gt_image, channel_index))
    background_recon = tensor_to_2d_numpy(channel_slice(reconstruction, channel_index))
    background_kwargs = {"cmap": "viridis", "vmin": 0.0, "vmax": 1.0}
    title_suffix = f"channel {channel_index}"
else:
    background_gt = np.clip(pseudorgb_gt, 0.0, 1.0)
    background_recon = np.clip(pseudorgb_reconstruction, 0.0, 1.0)
    background_kwargs = {}
    title_suffix = f"pseudoRGB bands [B,G,R] = {pseudorgb_bands}"

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(background_gt, **background_kwargs)
axes[0].scatter(x_px, y_px, s=scatter_size, c="cyan", alpha=scatter_alpha)
axes[0].set_title(f"Gaussian Centers on GT | {title_suffix}")
axes[0].axis("off")

axes[1].imshow(background_recon, **background_kwargs)
scatter = axes[1].scatter(
    x_px_high,
    y_px_high,
    s=scatter_size,
    c=strength_high,
    cmap="inferno",
    alpha=scatter_alpha,
    vmin=float(strength_high.min()),
    vmax=float(strength_high.max()),
)
axes[1].set_title(
    f"High-Weight Centers Colored by Mean Feature Strength | {title_suffix}\n"
    f"quantile >= {center_weight_quantile:.2f}, kept {high_weight_mask.sum()}/{high_weight_mask.size}"
)
axes[1].axis("off")
fig.colorbar(scatter, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

changed_idx = endmember_df["endmember_index"].head(topk_endmembers).tolist()
fig, axes = plt.subplots(len(changed_idx), 1, figsize=(12, 3 * len(changed_idx)), sharex=True)
if len(changed_idx) == 1:
    axes = [axes]

for ax, idx in zip(axes, changed_idx):
    idx = int(idx)
    ax.plot(E0.detach().cpu()[idx].numpy(), label="E0", linewidth=1.8)
    ax.plot(E_hat.detach().cpu()[idx].numpy(), label="E_hat", linewidth=1.8)
    ax.plot(delta_E.detach().cpu()[idx].numpy(), label="delta_E", linewidth=1.2, alpha=0.85)
    ax.set_title(f"Endmember {idx} | delta norm = {delta_row_norm[idx].item():.4f}")
    ax.grid(alpha=0.2)

axes[-1].set_xlabel("Spectral channel")
axes[0].legend()
plt.tight_layout()
plt.show()

gt_spectrum = gt_image[0, :, pixel_y, pixel_x].detach().cpu().numpy()
obs_spectrum = observed_image[0, :, pixel_y, pixel_x].detach().cpu().numpy()
recon_spectrum = reconstruction[0, :, pixel_y, pixel_x].detach().cpu().numpy()

plt.figure(figsize=(12, 4))
plt.plot(gt_spectrum, label="GT", linewidth=2.0)
plt.plot(obs_spectrum, label="Observed", linewidth=1.5, alpha=0.9)
plt.plot(recon_spectrum, label="Reconstruction", linewidth=1.5)
if display_mode == "single_band":
    plt.axvline(channel_index, color="gray", linestyle="--", alpha=0.6, label=f"selected channel {channel_index}")
else:
    marker_specs = [
        (pseudorgb_bands[0], "blue", "pseudoRGB blue band"),
        (pseudorgb_bands[1], "green", "pseudoRGB green band"),
        (pseudorgb_bands[2], "red", "pseudoRGB red band"),
    ]
    for band, color, label in marker_specs:
        plt.axvline(band, color=color, linestyle="--", alpha=0.6, label=label)
plt.title(f"Spectrum at pixel (y={pixel_y}, x={pixel_x})")
plt.xlabel("Spectral channel")
plt.ylabel("Intensity")
plt.grid(alpha=0.2)
plt.legend()
plt.show()